In [11]:
import numpy as np
import scipy as sp
import pandas as pd
import matplotlib.pyplot as plt
from StocProcess.RBM import MakeRBMTransProbFunc
from QAE.LowDepthQAE import LowDepthQAE

In [12]:
# RBM parameters
c = -1
d = 1
x0 = 0.5 * (c + d)
t0 = 0
t = 0.6
mu = 0.5
sigma = 1.0
n_terms = 5

# QAE setting
nShot = 12
epsilon = 0.0025
nRep = 10

In [13]:
np.random.seed(1)

In [14]:
# PDF at time t
transProbFunc = MakeRBMTransProbFunc(t, t0, c, d, mu, sigma, n_terms)

# Prob(X > (c + d)/2)
integFunc = lambda x: transProbFunc(x, x0)
pTrue, _ = sp.integrate.quad(integFunc, x0, 1.0)

In [15]:
Ns = (2 ** np.linspace(3, 7, 9)).astype(int)
print(Ns)

[  8  11  16  22  32  45  64  90 128]


In [16]:
retDf = pd.DataFrame(columns=['N', 'pTrue', 'beta', 'pEst', 'absErr', 'totalQueryNum', 'maxDepth'])

for _ in range(nRep):
    for iN in range(len(Ns)):
        N = Ns[iN]
        beta = np.log(N**0.5) / np.log(1 / epsilon)
        qaeRes = LowDepthQAE(pTrue, epsilon, nShot, beta)
        pEst = qaeRes.aEst
        totalQueryNum = qaeRes.TotalQueryNum * N * (N+1) / 2
        maxDepth = qaeRes.MaxDepth * N
        retDf.loc[len(retDf)] = [N, pTrue, beta, pEst, abs(pEst - pTrue), totalQueryNum, maxDepth]

c:\Users\koich\Desktop\Code\DivQCOSDE\QAE\MaximizeL.py:11: RuntimeWarning: divide by zero encountered in log
  neglogL = lambda theta: -np.dot(n1s, np.log(np.sin(thetaMuls * theta)**2)) - np.dot(n0s, np.log(np.cos(thetaMuls * theta)**2))


In [17]:
retDf

,N,pTrue,beta,pEst,absErr,totalQueryNum,maxDepth
0,8.0,0.649605,0.173534,0.649586,0.000018,516240.0,3000.0
1,11.0,0.649605,0.200109,0.650027,0.000422,1021680.0,3157.0
2,16.0,0.649605,0.231378,0.649326,0.000278,2134656.0,3216.0
3,22.0,0.649605,0.257954,0.649596,0.000008,5486052.0,3982.0
4,32.0,0.649605,0.289223,0.649926,0.000322,13330944.0,4512.0
...,...,...,...,...,...,...,...
85,32.0,0.649605,0.289223,0.649366,0.000238,13330944.0,4512.0
86,45.0,0.649605,0.317674,0.649906,0.000302,34428240.0,5535.0
87,64.0,0.649605,0.347067,0.649036,0.000568,83566080.0,6464.0
88,90.0,0.649605,0.375518,0.649736,0.000132,210073500.0,7650.0


In [18]:
retDf.groupby('N')['absErr'].mean()

N
8.0      0.000311
11.0     0.000382
16.0     0.000298
22.0     0.000279
32.0     0.000347
45.0     0.000230
64.0     0.000329
90.0     0.000190
128.0    0.000277
Name: absErr, dtype: float64

In [21]:
retDf.to_csv('RBM_LowDepthQAE.csv', index=False)